# Setup

In [74]:
# Fabric notebook parameter. The tumbling window trigger passes its window start
# as a full ISO timestamp ("2026-08-25T02:00:00Z"), so only the date part is
# usable as a partition value.
#
# This cell must stay tagged `parameters` — Fabric injects the pipeline's value
# in a new cell directly below it, so this is a default, not a constant.
ingestion_date = None

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 121, Finished, Available, Finished, False)

In [75]:
# Environment bootstrap. A Fabric notebook has neither the repo root on
# sys.path nor as its working directory, so relative paths like
# "config/config.yaml" cannot resolve there. Detecting the OneLake mount keeps
# one notebook working in both places instead of maintaining two copies.
import os
import sys

CODE_ROOT = "/lakehouse/default/Files/code"
IN_FABRIC = os.path.isdir(CODE_ROOT)

if IN_FABRIC and CODE_ROOT not in sys.path:
    sys.path.append(CODE_ROOT)

CONFIG_DIR = f"{CODE_ROOT}/config" if IN_FABRIC else "config"

# In Fabric, secrets come from the workspace rather than a gitignored .env.
# Set them here for a trial run; Chapter 8 replaces this with Key Vault.
if IN_FABRIC:
    os.environ.setdefault("APP_ENV", "fabric")


print("Running in Fabric" if IN_FABRIC else "Running locally")

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 122, Finished, Available, Finished, False)

Running in Fabric


In [76]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from transformation.bronze_to_silver import run_silver_transformation
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger

setup_logging(config_path=f"{CONFIG_DIR}/logging_config.yaml")
logger = get_logger(__name__)

# Fabric provides a Delta-enabled session already; getOrCreate() returns it.
spark = SparkSession.builder.appName("silver_cleaning").getOrCreate()

app_config = load_config(
    config_path=f"{CONFIG_DIR}/config.yaml",
    sources_path=f"{CONFIG_DIR}/sources.yaml",
)

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 123, Finished, Available, Finished, False)

# Distribute the code to Spark executors

In [77]:
# Ship the code to the EXECUTORS. sys.path above only affects the driver, but
# Spark's Python workers are separate processes with their own interpreter and
# path. cloudpickle serializes module-level UDFs by reference ("import
# transformation.salary_parser, get this attribute"), so every worker has to be
# able to import the package or deserialization fails with ModuleNotFoundError
# — surfacing at the first action (.count(), .show()), not where the UDF was
# defined. addPyFile distributes the zip and adds it to each worker's path.
if IN_FABRIC:
    import shutil

    PKG_ZIP = shutil.make_archive("/tmp/jma_code", "zip", CODE_ROOT)
    try:
        spark.sparkContext.addPyFile(PKG_ZIP)
        print("Shipped", PKG_ZIP, "to executors")
    except Exception as exc:
        # Re-running in the same session hits a name collision, which is
        # harmless: the file is already registered from the first call.
        print("addPyFile skipped:", exc)

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 124, Finished, Available, Finished, False)

Shipped /tmp/jma_code.zip to executors


# Paths

In [78]:
# Tables/dbo/, not Tables/ — this Lakehouse is schema-enabled, so Fabric
# reads Tables/<name>/ as a SCHEMA named <name>. Writing to "Tables/x"
# therefore registers a schema called x with loose Delta files inside it,
# not a table. Spark still reads it by path, which is why nothing broke,
# but the SQL analytics endpoint and DirectLake only see catalog-
# registered tables — so Power BI would find nothing (Chapter 7).
from datetime import date

# Parameterised rather than hardcoded to today so a backfill run processes
# the window it was given instead of silently reprocessing today.
INGESTION_DATE = (ingestion_date or str(date.today()))[:10]
print("Processing ingestion_date:", INGESTION_DATE)

if IN_FABRIC:
    BRONZE_TABLE_PATH = "Tables/dbo/bronze_job_postings"
    SILVER_TABLE_PATH = "Tables/dbo/silver_job_postings"
    QUARANTINE_TABLE_PATH = "Tables/dbo/silver_job_postings_quarantine"
    REFERENCE_PATH = "Files/code/config/reference/country_codes.csv"
else:
    BRONZE_TABLE_PATH = "data/delta/bronze_job_postings"
    SILVER_TABLE_PATH = "data/delta/silver_job_postings"
    QUARANTINE_TABLE_PATH = "data/delta/silver_job_postings_quarantine"
    REFERENCE_PATH = "config/reference/country_codes.csv"

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 125, Finished, Available, Finished, False)

Processing ingestion_date: 2026-08-28


# Run Bronze -> Silver for today's partition

This is the only cell that writes the Silver table. Skipping it leaves
`PATH_NOT_FOUND` on every cell below, which reads as a path bug but is not one.

In [79]:
run_silver_transformation(
    spark=spark,
    bronze_table_path=BRONZE_TABLE_PATH,
    reference_path=REFERENCE_PATH,
    silver_table_path=SILVER_TABLE_PATH,
    quarantine_table_path=QUARANTINE_TABLE_PATH,
    ingestion_date=INGESTION_DATE,
)

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 126, Finished, Available, Finished, False)

2026-08-28 21:38:51 | INFO     | run_id=5ea9a238 | transformation.bronze_to_silver | Deduplication: removed 753 duplicate postings across sources
2026-08-28 21:38:53 | INFO     | run_id=5ea9a238 | transformation.dq_checks | DQ engine: 547 rows passed, 0 rows quarantined (critical rule failures)
2026-08-28 21:39:01 | INFO     | run_id=5ea9a238 | transformation.bronze_to_silver | Wrote 547 clean rows to Silver: Tables/silver_job_postings


# Inspect results

If most rows land in quarantine, a DQ rule is too strict — that is a
tuning decision, not a bug.

In [80]:
silver_df = spark.read.format("delta").load(SILVER_TABLE_PATH)
print("Silver row count:", silver_df.count())
silver_df.groupBy("source").count().show()
silver_df.filter(F.size(F.col("dq_warnings")) > 0).select(
    "title", "company", "dq_warnings"
).show(10, truncate=60)

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 127, Finished, Available, Finished, False)

Silver row count: 697
+------+-----+
|source|count|
+------+-----+
|jooble|  352|
|adzuna|  345|
+------+-----+

+------------------------------+---------------------+---------------------+
|                         title|              company|          dq_warnings|
+------------------------------+---------------------+---------------------+
|   Marketing Manager (Protein)|                 NULL|    [company_present]|
|       Operations Manager (FP)|                 NULL|    [company_present]|
|       Sales Manager (Protein)|                 NULL|    [company_present]|
|       Brand Manager (Protein)|                 NULL|    [company_present]|
|    VP Manufacturing - Protein|                 NULL|    [company_present]|
|           Compliance Director|Crowley for Wisconsin|[description_present]|
|Financial Controller (Protein)|                 NULL|    [company_present]|
|  Production Manager - Poultry|                 NULL|    [company_present]|
|           Plant Manager (RTE)|        

# Quarantined rows

Rows that failed a critical DQ rule. An empty table here is good; a
quarantine larger than Silver means a rule is misconfigured, not that
the source data is unusable.

In [81]:
from delta.tables import DeltaTable

# The pipeline only creates this table when a row actually fails a critical
# rule, so a missing path means zero quarantined rows — the good case, not an
# error. Checking first keeps a clean run from raising PATH_NOT_FOUND.
if not DeltaTable.isDeltaTable(spark, QUARANTINE_TABLE_PATH):
    print("No quarantine table yet — no rows have failed a critical DQ rule.")
else:
    quarantine_df = spark.read.format("delta").load(QUARANTINE_TABLE_PATH)
    quarantine_count = quarantine_df.count()
    print("Quarantined row count:", quarantine_count)

    # Note: apply_dq_rules drops its _failed_<rule> diagnostic columns before
    # the split, so the table records THAT a row failed a critical rule but not
    # WHICH one. Inspect the row values against build_dq_rules to work it out.
    if quarantine_count > 0:
        quarantine_df.select(
            "source", "title", "company", "salary_min", "posted_date", "dq_warnings"
        ).show(10, truncate=60)

StatementMeta(, 35ec7a6c-d32d-4341-b79d-205fb669eac8, 128, Finished, Available, Finished, False)

No quarantine table yet — no rows have failed a critical DQ rule.
